# 1. Introducción: reingreso hospitalario antes de 30 días

Este Jupyter Book corresponde al **Primer Entregable** del proyecto de Machine
Learning y sigue la ruta de clasificación. Utiliza *Diabetes 130-US Hospitals
for Years 1999–2008*, publicado por UCI {cite}`uci_diabetes_130_2014`, y la
documentación del estudio original de Strack et al. {cite}`strack_2014`.

## Problema y pregunta de investigación

**Pregunta:** ¿Es posible predecir, utilizando información disponible durante
la hospitalización y hasta el momento inmediatamente anterior al alta, qué
pacientes con diabetes serán readmitidos al hospital dentro de los 30 días
posteriores al alta?

El resultado puede apoyar la priorización de seguimiento, coordinación y
revisión clínica, pero este proyecto es académico y no constituye una
herramienta clínica lista para implementación.

## Variable objetivo

`readmitted` tiene tres valores: `<30`, `>30` y `NO`. El objetivo derivado es:

- `<30` → `reingreso_30d = 1`;
- `>30` y `NO` → `reingreso_30d = 0`.

Esta definición responde directamente a la relevancia metodológica del
**reingreso temprano**. No se eligió para balancear artificialmente las clases;
por ello el desbalance y sus consecuencias se estudian explícitamente.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

## Fuente, licencia y alcance del dataset

La ficha oficial de UCI informa 101.766 encuentros y 50 columnas originales,
provenientes de 130 hospitales y redes integradas de Estados Unidos durante
1999–2008. La unidad de observación es un **encuentro hospitalario** de un
paciente con diabetes; un paciente puede aportar más de un encuentro. La
población incluye hospitalizaciones de 1 a 14 días con pruebas de laboratorio
y medicación administrada.

- **Autores del recurso UCI:** John Clore, Krzysztof Cios, Jonathan DeShazo y
  Beata Strack.
- **DOI:** `10.24432/C5230J`.
- **Licencia:** CC BY 4.0.
- **Archivos:** `diabetic_data.csv` e `IDS_mapping.csv`.

El proyecto usa primero los CSV incluidos en `data/`. Si falta el archivo
principal, intenta descargar el ZIP oficial de UCI; si falla, emite un error
que indica exactamente qué archivos copiar. No se emplean rutas absolutas.

In [2]:
raw = load_raw()
n_rows, n_cols = raw.shape
md_out(f"- **Filas (encuentros):** {es(n_rows, 0)}\n- **Columnas:** {n_cols} (incluye el objetivo derivado `{TARGET}`)\n"
       f"- **Pacientes distintos:** {es(raw[GROUP_COL].nunique(), 0)} (un paciente puede tener varios encuentros)\n"
       f"- **Mínimo del enunciado:** 20.000 observaciones → {'cumple' if n_rows >= 20000 else 'NO cumple'}")
raw.head(3).T.head(12)

- **Filas (encuentros):** 101.766
- **Columnas:** 51 (incluye el objetivo derivado `reingreso_30d`)
- **Pacientes distintos:** 71.518 (un paciente puede tener varios encuentros)
- **Mínimo del enunciado:** 20.000 observaciones → cumple

,0,1,2
encounter_id,2278392,149190,64410
patient_nbr,8222157,55629189,86047875
race,Caucasian,Caucasian,AfricanAmerican
gender,Female,Female,Female
age,[0-10),[10-20),[20-30)
weight,NaN,NaN,NaN
admission_type_id,6,1,1
discharge_disposition_id,25,1,1
admission_source_id,1,7,7
time_in_hospital,1,3,2


## Elección de la ruta del enunciado

El enunciado distingue cuatro rutas según el tipo de dataset: **A** clasificación, **B** regresión sin tiempo ni coordenadas,
**C** regresión temporal y **D** espacio-temporal. Las secciones 2.6 a 2.8 (temporal y espacial) son obligatorias **solo si hay fecha
u hora o latitud y longitud**. Se comprueba buscando esas columnas.

In [3]:
fechas = [c for c in raw.columns if re.search(r"(date|fecha|timestamp|time_stamp)", c, re.I)]
coords = [c for c in raw.columns if re.search(r"(^|_)(lat|lon|lng|latitude|longitude|coord|zip|zipcode|city|state)($|_)", c, re.I)]
md_out(f"- Columnas con fecha u hora: **{fechas if fechas else 'ninguna'}**\n"
       f"- Columnas con coordenadas o ubicación: **{coords if coords else 'ninguna'}**\n\n"
       + ("**Conclusión:** no hay fecha ni coordenadas, así que se sigue la **ruta A (clasificación)** sin las secciones 2.6 a 2.8. "
          "El dataset abarca 1999-2008, pero **no** trae la fecha de cada encuentro; por eso no puede analizarse deriva temporal "
          "(se declara como limitación en el capítulo 10)." if not fechas and not coords else
          "**Atención:** hay columnas que podrían ser fecha o ubicación; revísalas antes de continuar."))

- Columnas con fecha u hora: **ninguna**
- Columnas con coordenadas o ubicación: **ninguna**

**Conclusión:** no hay fecha ni coordenadas, así que se sigue la **ruta A (clasificación)** sin las secciones 2.6 a 2.8. El dataset abarca 1999-2008, pero **no** trae la fecha de cada encuentro; por eso no puede analizarse deriva temporal (se declara como limitación en el capítulo 10).

## Partición entrenamiento / prueba

El enunciado exige **reservar el conjunto de prueba antes de cualquier decisión** (imputación, transformaciones, selección de
variables, umbrales) y que la partición respete la estructura de los datos: **por grupos si hay varias filas por entidad**.

Aquí un mismo paciente puede aparecer en varios encuentros, y sus resultados se parecen entre sí (se cuantifica en el capítulo 7).
Si se separaran los encuentros al azar, un paciente quedaría en entrenamiento y en prueba a la vez y la evaluación sería optimista.
Por eso la partición es **80/20 estratificada por el objetivo de reingreso <30 días y agrupada por paciente** (`StratifiedGroupKFold`, semilla 42, un
pliegue de cinco como prueba). El archivo es único (no viene separado en entrenamiento y prueba), de modo que la partición la
define este libro y queda guardada en `tables/particion_encuentros.csv`.

In [4]:
train_all, test_all = load_train(), load_test()
overlap = set(train_all[GROUP_COL]) & set(test_all[GROUP_COL])
assert len(overlap) == 0, "Hay pacientes en entrenamiento y prueba a la vez"

resumen = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Prueba"],
    "encuentros": [len(train_all), len(test_all)],
    "pacientes": [train_all[GROUP_COL].nunique(), test_all[GROUP_COL].nunique()],
    "reingreso_30d_%": [100 * train_all[TARGET].mean(), 100 * test_all[TARGET].mean()],
})
pd.concat([train_all[[ID_COL]].assign(conjunto="train"), test_all[[ID_COL]].assign(conjunto="test")]) \
  .to_csv(TABLES / "particion_encuentros.csv", index=False)
display(resumen.round(2))
md_out(f"Pacientes en ambos conjuntos: **{len(overlap)}**. La prevalencia del objetivo es similar en los dos conjuntos "
       f"({es(train_all[TARGET].mean(), 1, pct=True)} frente a {es(test_all[TARGET].mean(), 1, pct=True)}), como se esperaba de una partición estratificada. "
       "A partir de aquí, **todo el EDA y todas las decisiones usan solo el entrenamiento**; la prueba se toca de nuevo en el capítulo 9.")

,conjunto,encuentros,pacientes,reingreso_30d_%
0,Entrenamiento,81413,57231,11.08
1,Prueba,20353,14287,11.47


Pacientes en ambos conjuntos: **0**. La prevalencia del objetivo es similar en los dos conjuntos (11,1 % frente a 11,5 %), como se esperaba de una partición estratificada. A partir de aquí, **todo el EDA y todas las decisiones usan solo el entrenamiento**; la prueba se toca de nuevo en el capítulo 9.

## Mapa de capítulos

| Capítulo | Contenido | Sección del enunciado |
|---|---|---|
| 2 | Base de datos: calidad, faltantes, duplicados, sesgos, ética | Base de datos (10 %) |
| 3 | Variable objetivo | 2.1 |
| 4 | Análisis univariado | 2.2 |
| 5 | Análisis bivariado | 2.3 |
| 6 | Análisis multivariado | 2.4 |
| 7 | Auditoría de fuga de datos | 2.5 |
| 8 | Preprocesamiento y Pipeline | 2.9 |
| 9 | Modelo base (Dummy y regresión logística) | Modelo base (30 %) |
| 10 | Conclusiones y checklist | — |

## Consideraciones éticas (resumen)

Los datos son registros clínicos anonimizados de pacientes reales. Incluyen raza, sexo y edad; un modelo de riesgo podría
reproducir desigualdades de acceso a la atención si se usara sin cuidado. El capítulo 2 discute el riesgo de reidentificación y el
capítulo 10 las limitaciones y usos apropiados.